## Importando bibliotecas

In [1]:
import tensorflow as tf
import keras as ke
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn.metrics as skm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import seaborn as sns
from tqdm import tqdm
import os
import cv2

%config Completer.use_jedi = False


import warnings
warnings.simplefilter("ignore")

## Definindo Cores do Hitmap

In [2]:
colors_dark = ["#1F1F1F", "#313131", '#636363', '#AEAEAE', '#DADADA']
colors_red = ["#331313", "#582626", '#9E1717', '#D35151', '#E9B4B4']
colors_blue = ['#008B8B','#5F9EA0','#20B2AA','#66CDAA','#7FFFD4']

sns.palplot(colors_dark)
sns.palplot(colors_blue)
sns.palplot(colors_red)

## Carregar e converter imagens

In [3]:
X_train = []
y_train = []

X_test = []
y_test = []


path_train = ('/content/drive/MyDrive/visaocomputacional/avaliacao/Training/')
path_test = ('/content/drive/MyDrive/visaocomputacional/avaliacao/Testing/')
img_size = 150

for i in os.listdir(path_train):
    for j in os.listdir(path_train+i):
        X_train.append (cv2.resize(cv2.imread(path_train+i+'/'+j), (img_size,img_size)))
        y_train.append(i)

for i in os.listdir(path_test):
    for j in os.listdir(path_test+i):
        X_test.append (cv2.resize(cv2.imread(path_test+i+'/'+j), (img_size,img_size)))
        y_test.append(i)

X_train = (np.array(X_train))
X_test = (np.array(X_test))


train_labels_encoded = [0 if category == 'no_tumor' else(1 if category == 'glioma_tumor' else(2 if category=='meningioma_tumor' else 3)) for category in list(y_train)]
test_labels_encoded = [0 if category == 'no_tumor' else(1 if category == 'glioma_tumor' else(2 if category=='meningioma_tumor' else 3)) for category in list(y_test)]

## Verificando as classes

In [4]:
classes = np.unique(y_train)
nClasses = len(classes)
print('Total number of outputs : ', nClasses)
print('Output classes : ', classes)

plt.figure(figsize=[10,10])

plt.subplot(121)
plt.imshow(X_train[500,:,:], cmap=plt.cm.binary)
plt.title("Ground Truth : {}".format(y_train[500]))

plt.subplot(122)
plt.imshow(X_test[20,:,:], cmap=plt.cm.binary)
plt.title("Ground Truth : {}".format(y_test[20]))

In [5]:
plt.figure(figsize=[10,10])
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i], cmap=plt.cm.binary)
    plt.xlabel(y_train[i])
plt.show()

In [6]:
plt.figure(figsize = (17,8));
lis = ['Train', 'Test']
for i,j in enumerate([y_train, y_test]):
    plt.subplot(1,2, i+1);
    sns.countplot(x = j);
    plt.xlabel(lis[i])

In [7]:
np.random.RandomState(4)
tf.random.set_seed(4)

model = tf.keras.Sequential(
        [
          tf.keras.layers.Conv2D(kernel_size=(5,5) ,filters=32, activation='relu', padding='same'),
          tf.keras.layers.MaxPool2D(pool_size=(2,2)),

          tf.keras.layers.Conv2D(kernel_size=(3,3),filters=64, activation='relu', padding='same'),
          tf.keras.layers.MaxPool2D(pool_size=(2,2)),

          tf.keras.layers.Conv2D(kernel_size=(3,3) ,filters=128, activation='relu', padding='same'),
          tf.keras.layers.MaxPool2D(pool_size=(2,2)),

          tf.keras.layers.Conv2D(kernel_size=(3,3) ,filters=256, activation='relu', padding='same'),
          tf.keras.layers.MaxPool2D(pool_size=(2,2)),

          tf.keras.layers.Dense(128, activation='relu'),
          tf.keras.layers.Flatten(),
          tf.keras.layers.Dense(256, activation='relu'),
          tf.keras.layers.Dropout(rate=0.4),
          tf.keras.layers.Dense(4, activation='softmax')
  ])
model.compile(optimizer=tf.keras.optimizers.Adam(weight_decay=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
history = model.fit(
    tf.cast(X_train, tf.float32), np.array(pd.get_dummies(y_train)),
    validation_split=0,
    epochs = 15,
    verbose=1,
    batch_size=128)

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
plt.plot(history.history["loss"], label="treino")
plt.plot(history.history["val_loss"], label="validacao")
plt.xlabel("epocas")
plt.ylabel("Loss")
#plt.ylim([0.0, 0.68])
plt.legend();
plt.subplot(1,2,2)
plt.plot(history.history["accuracy"], label="treino")
plt.plot(history.history["val_accuracy"], label="validacao")
plt.xlabel("epocas")
plt.ylabel("acurácia")
#plt.ylim([0.75, 0.94])
plt.legend();

In [ ]:
predicted_classes = model.predict(X_test)
predicted_classes = np.argmax(np.round(predicted_classes),axis=1)

target_names = ["Class {}".format(i) for i in range(4)]

print(classification_report(test_labels_encoded, predicted_classes, target_names=target_names))

In [ ]:
from sklearn.metrics import confusion_matrix

cmat = confusion_matrix(test_labels_encoded, predicted_classes)

# Criando um dataframe para a matriz de confusão formatada em array, para que seja fácil de plotar.

cmat_df = pd.DataFrame(cmat,
                     index = ['0','1','2','3'],
                     columns = ['0','1','2','3'])

#Plotando:
plt.figure(figsize=(10,6))
sns.heatmap(cmat_df, annot=True,fmt="d", cmap=colors_red[::-1],alpha=0.7,linewidths=2,linecolor=colors_dark[3])
plt.title('Matriz de Confusão')
plt.ylabel('Valores atuais')
plt.xlabel('Valores preditos')
plt.show()